# Easy Italian News
- download and 

# Find all days with news

In [26]:
import requests
from bs4 import BeautifulSoup
import re

l_to_remove = [#r'\n+', 
    r'www\..*\n',
    r'Click the link..*\n', 
    r'https://..*\n', 
    r'\xa0\n', 
    r'Creative Commons..*\n',
    r'Image.*\n', 
    # r'it\.[\w+]..*\n', 'tg24\..*\n',
    # r'[\s]Jeffrey Zeldman\n',
    # r'[\s]Zdravko Petrov\n', 
    # r'[\s]Chris Watt\n',
    # r'[\s]GovernmentZA.*\n',
    # r'[\s]Attribution..*\n',
    # r'[\s]Elvert Barnes..*\n',
    # r'[\s]Maritza Ríos..*\n', 
    # r'[\s]si.robi.*\n',
    # r'[\s]Steven Depolo.*\n',
    # r'[\s]Brendan Keene.*\n',
    # r'[\s]Francesco Ranieri.*\n',
    # r'[\s]Nathan Keirn.*\n',
    # r'[\s]Giorgio Minguzzi.*\n',
    #r'^[\w+]\.[\w+]\.[\w+]$'
    #r'^\s*[A-Za-z0-9_-]+(?:\.[A-Za-z0-9_-]+){1,3}\s*$',
    
    r' \n',    
]

# Pattern to remove 2-4 words separated by dots.
# The only space can be at the beginnign of the line
pattern = re.compile(
    r'^\s*[A-Za-z0-9_-]+(?:\.[A-Za-z0-9_-]+){1,3}\s*$',
    re.MULTILINE
)

# Specify the URL of the website you want to scrape
url = 'https://easyitaliannews.com/2026/06/'

# To avoid server error: 403
headers = {
    "User-Agent": "Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:109.0) Gecko/20100101 Firefox/117.0"
}

# Send a GET request to the URL
response = requests.get(url, headers=headers)

# Check if the request was successful (status code 200)
if response.status_code == 200:
    # Parse the HTML content using BeautifulSoup
    soup = BeautifulSoup(response.text, 'html.parser')

    l_tds = soup.find_all('td')

    l_of_news = []
    for td in soup.find_all('td'):
        try:
            l_of_news.append(td.find('a').get('href'))
        except:
            pass

In [27]:
l_of_news

['https://easyitaliannews.com/2026/06/02/',
 'https://easyitaliannews.com/2026/06/04/',
 'https://easyitaliannews.com/2026/06/06/',
 'https://easyitaliannews.com/2026/06/09/',
 'https://easyitaliannews.com/2026/06/11/',
 'https://easyitaliannews.com/2026/06/13/']

In [29]:
for url in l_of_news:
#for url in l_of_news[:1]:
    news_date = url[28:38].replace('/', '-')
    print(news_date)

    # Send a GET request to the URL
    response = requests.get(url, headers=headers)

    # Check if the request was successful (status code 200)
    if response.status_code == 200:
        # Parse the HTML content using BeautifulSoup
        soup = BeautifulSoup(response.text, 'html.parser')

        ns = soup.find('div', {'class': 'entry-content'})

        # Extracting all paragraphs
        l_content = []
        mp3 = ''
        paragraphs = ns.find_all(True)
        for idx, paragraph in enumerate(paragraphs):
            # print(f"Paragraph {idx+1}: {paragraph.text}")
            p = str(paragraph.text)
        #     if  p not in l_content:
        #         l_content.append(p)
            if mp3 == '' and 'mp3' in p:
                mp3 = p.strip()
            l_content.append(p)

        s = ('\n').join(l_content).split('Il tuo aiuto per noi è importante!')[0].split('Subscribe')[0]


        t = ''
        for el in l_to_remove:
            t = re.sub(el, '\n', s)
            s = t

        # Remove the links to websites (2-4 words separate by dots)
        cleaned_text = pattern.sub('', s)
        
        # Replace 2 or more consecutive blank lines with a single blank line
        text = re.sub(r'\n\s*\n+', '\n\n', cleaned_text)

        # 
        s = re.sub(r'^(.*)(\r?\n)\1(\r?\n)?', r'\1\2', text, flags=re.MULTILINE)
        
        # t = re.sub(r'\n\n+', '\n\n', s)
        # s = t
        # s
        # s = s.strip()

        # f = open('output.txt', 'w')
        # f.write(s)
        # f.close()

        # with open('text.txt', 'w', encoding='utf-8') as f:
        #     f.write(s)

        with open(f'EasyItalianNews_{news_date}.txt', 'w', encoding='utf-8') as f:
            f.write(s)

        doc = requests.get(mp3)

        with open(f'EasyItalianNews_{news_date}.mp3', 'wb') as f:
            f.write(doc.content)
print('done')

2026-06-02
2026-06-04
2026-06-06
2026-06-09
2026-06-11
2026-06-13
done
